# Polynomial Regression: SVD Optimizer Analysis

Analysis of SVD optimizer vs standard optimizers on polynomial regression task.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from style import set_style, lr_labels
set_style()

# SVD vs baseline - full torch SVD + truncation

In [ ]:
# Load JSONL results
from style import load_results, load_results_jsonl

PLOT_DIR = Path('plots/polynomial')
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOT_DIR.resolve()}")

df = load_results("polynomial_scan")
print(f"Total: {len(df)} experiment runs")
print(f"Optimizers: {sorted(df['optimizer'].unique())}")
print(f"Batch sizes: {sorted(df['batch_size'].unique())}")

def is_bad(row):
    if np.isnan(row['losses']['val'][-1]):
        return True
    return False
df['is_bad'] = df.apply(is_bad, axis=1)
df = df[~df['is_bad']]
print(f"After removing bad runs: {len(df)} experiment runs")

In [ ]:
SEED_SEL = 5310
BASELINE_OPT = ['SGD','PolyakSGD','RMSprop','Adam','LBFGS','JD_UPGrad','HIG']
df = df[df.model_seed == SEED_SEL]

In [ ]:
# Helper functions
def get_final_loss(row, loss_type='val'):
    losses = row['losses'][loss_type]
    # Return last non-NaN value (handles LBFGS instability)
    for val in reversed(losses):
        if val is not None and not (isinstance(val, float) and np.isnan(val)):
            return val
    return np.nan

def get_loss_curve(row, loss_type='val'):
    return np.array(row['losses'][loss_type])

def sliding_average(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

# Add derived columns
df['final_val_loss'] = df.apply(lambda r: get_final_loss(r, 'val'), axis=1)
df['final_train_loss'] = df.apply(lambda r: get_final_loss(r, 'train'), axis=1)
df['total_time'] = df['losses'].apply(lambda l: l.get('total_time', np.nan))
df['avg_epoch_time'] = df['losses'].apply(lambda l: l.get('avg_epoch_time', np.nan))
df['avg_batch_time_train'] = df['losses'].apply(lambda l: l.get('avg_batch_time_train', np.nan))

df_svd = df[(df['optimizer'] == 'SVD')].copy()
df_baseline = df[df['optimizer'] != 'SVD'].copy()

bs = sorted(df['batch_size'].unique())[0]  # 32
baseline_optimizers = sorted(df_baseline['optimizer'].unique().tolist())
k_fractions = sorted(df_svd['k_fraction'].dropna().unique())
svd_lrs = sorted(df_svd['lr'].unique())
svd_rtols = sorted(df_svd['rtol'].dropna().unique())

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
n_epochs = len(train_curve)
epochs_train = np.arange(1,n_epochs + 1)

ax.plot(epochs_train, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]
    label = f"{opt}"
    ax.plot(epochs_train, get_loss_curve(best_row, 'train'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,5])
plt.xlim(0,n_epochs+1)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'train_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'val')
n_epochs = len(train_curve)
epochs_train = np.arange(n_epochs)

ax.plot(epochs_train, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]

    label = f"{opt}"
    ax.plot(epochs_train, get_loss_curve(best_row, 'val'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
plt.xlim(0,n_epochs)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'val_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
wall_times = get_loss_curve(best_svd_row, 'epoch_times')
t = np.cumsum(wall_times)

ax.plot(t, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]
    t = get_loss_curve(best_row, 'epoch_times')
    t = np.cumsum(t)
    label = f"{opt}"
    ax.plot(t, get_loss_curve(best_row, 'train'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Wall Time (s)')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
#plt.xlim([1,None])
plt.xscale('log')
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_train_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'val')
wall_times = get_loss_curve(best_svd_row, 'epoch_times')
wall_times = np.concatenate([np.array([1]), np.array(wall_times)])
t = np.cumsum(wall_times)

ax.plot(t, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]
    t = get_loss_curve(best_row, 'epoch_times')
    t = np.concatenate([np.array([1]), np.array(t)])
    t = np.cumsum(t)
    #label = f"{opt} ($\eta={lr_labels[best_row['lr']]}$)" if opt != "PolyakSGD" else f"{opt}"
    label = f"{opt}"
    ax.plot(t, get_loss_curve(best_row, 'val'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Wall Time + 1 (sec)')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
plt.xlim([1,None])
plt.xscale('log')
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_val_loss_best.pdf')
plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
    for RTOL in [1e-4,1e-3,1e-2]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            train_curve = get_loss_curve(row, 'train')
            epochs = np.arange(1, len(train_curve) + 1)
            ax.plot(epochs, train_curve, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        best_sgd_row = df.loc[df_sgd['final_val_loss'].idxmin()]
        ax.plot(epochs, get_loss_curve(best_sgd_row, 'train'), label=f"SGD ($\eta={lr_labels[best_sgd_row['lr']]}$)", color='k', linestyle='--', linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Train Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, 1D Random Polynomial ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.ylim([None,3])
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"train_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
    for RTOL in [1e-4,1e-3,1e-2]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            train_curve = get_loss_curve(row, 'val')
            epochs = np.arange(len(train_curve))
            ax.plot(epochs, train_curve, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        best_sgd_row = df.loc[df_sgd['final_val_loss'].idxmin()]
        ax.plot(epochs, get_loss_curve(best_sgd_row, 'val'), label=f"SGD ($\eta={lr_labels[best_sgd_row['lr']]}$)", color='k', linestyle='--', linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, Polynomial ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.ylim([None,3])
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"val_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

# SVD vs baseline - randomized $k$-SVD

In [ ]:
# Load JSONL results
from style import load_results, load_results_jsonl

PLOT_DIR = Path('plots/polynomial_SVDrand')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

df = load_results("polynomial_scan")

def is_bad(row):
    if np.isnan(row['losses']['val'][-1]):
        return True
    return False
df['is_bad'] = df.apply(is_bad, axis=1)
df = df[~df['is_bad']]
df_baseline = df[df['optimizer'] != 'SVD'].copy()
del df

## load new SVD results
df_svd = load_results("polynomial_scan_SVDrand")

In [ ]:
SEED_SEL = 5310
BASELINE_OPT = ['SGD','PolyakSGD','RMSprop','Adam','LBFGS','JD_UPGrad','HIG']
df_baseline = df_baseline[df_baseline.model_seed == SEED_SEL]
df_svd = df_svd[df_svd.model_seed == SEED_SEL]

In [ ]:
# Helper functions
def get_final_loss(row, loss_type='val'):
    losses = row['losses'][loss_type]
    # Return last non-NaN value (handles LBFGS instability)
    for val in reversed(losses):
        if val is not None and not (isinstance(val, float) and np.isnan(val)):
            return val
    return np.nan

def get_loss_curve(row, loss_type='val'):
    return np.array(row['losses'][loss_type])

def sliding_average(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

# Add derived columns
df_baseline['final_val_loss'] = df_baseline.apply(lambda r: get_final_loss(r, 'val'), axis=1)
df_baseline['final_train_loss'] = df_baseline.apply(lambda r: get_final_loss(r, 'train'), axis=1)
df_baseline['total_time'] = df_baseline['losses'].apply(lambda l: l.get('total_time', np.nan))
df_baseline['avg_epoch_time'] = df_baseline['losses'].apply(lambda l: l.get('avg_epoch_time', np.nan))
df_baseline['avg_batch_time_train'] = df_baseline['losses'].apply(lambda l: l.get('avg_batch_time_train', np.nan))

df_svd['final_val_loss'] = df_svd.apply(lambda r: get_final_loss(r, 'val'), axis=1)
df_svd['final_train_loss'] = df_svd.apply(lambda r: get_final_loss(r, 'train'), axis=1)
df_svd['total_time'] = df_svd['losses'].apply(lambda l: l.get('total_time', np.nan))
df_svd['avg_epoch_time'] = df_svd['losses'].apply(lambda l: l.get('avg_epoch_time', np.nan))
df_svd['avg_batch_time_train'] = df_svd['losses'].apply(lambda l: l.get('avg_batch_time_train', np.nan))

bs = sorted(df_svd['batch_size'].unique())[0]  # 32
baseline_optimizers = sorted(df_baseline['optimizer'].unique().tolist())
k_fractions = sorted(df_svd['k_fraction'].dropna().unique())
svd_lrs = sorted(df_svd['lr'].unique())
svd_rtols = sorted(df_svd['rtol'].dropna().unique())

In [ ]:
colors = sns.color_palette("bright", n_colors=len(baseline_optimizers))

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
n_epochs = len(train_curve)
epochs_train = np.arange(1,n_epochs + 1)

ax.plot(epochs_train, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]
    label = f"{opt}"
    ax.plot(epochs_train, get_loss_curve(best_row, 'train'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,5])
plt.xlim(0,n_epochs+1)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'train_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("bright", n_colors=len(baseline_optimizers))

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'val')
n_epochs = len(train_curve)
epochs_train = np.arange(n_epochs)

ax.plot(epochs_train, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]

    label = f"{opt}"
    ax.plot(epochs_train, get_loss_curve(best_row, 'val'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
plt.xlim(0,n_epochs)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'val_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("bright", n_colors=len(baseline_optimizers))

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
wall_times = get_loss_curve(best_svd_row, 'epoch_times')
t = np.cumsum(wall_times)

ax.plot(t, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]
    t = get_loss_curve(best_row, 'epoch_times')
    t = np.cumsum(t)
    label = f"{opt}"
    ax.plot(t, get_loss_curve(best_row, 'train'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Wall Time (s)')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
#plt.xlim([1,None])
plt.xscale('log')
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_train_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("bright", n_colors=len(baseline_optimizers))

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'val')
wall_times = get_loss_curve(best_svd_row, 'epoch_times')
wall_times = np.concatenate([np.array([1]), np.array(wall_times)])
t = np.cumsum(wall_times)

ax.plot(t, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]
    t = get_loss_curve(best_row, 'epoch_times')
    t = np.concatenate([np.array([1]), np.array(t)])
    t = np.cumsum(t)
    #label = f"{opt} ($\eta={lr_labels[best_row['lr']]}$)" if opt != "PolyakSGD" else f"{opt}"
    label = f"{opt}"
    ax.plot(t, get_loss_curve(best_row, 'val'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Wall Time + 1 (sec)')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
plt.xlim([1,None])
plt.xscale('log')
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_val_loss_best.pdf')
plt.show()

In [ ]:
for LR in [0.05,0.1,0.5,1.0]:
    for RTOL in [1e-4,1e-3,1e-2]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            train_curve = get_loss_curve(row, 'train')
            epochs = np.arange(1, len(train_curve) + 1)
            ax.plot(epochs, train_curve, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        best_sgd_row = df_baseline.loc[df_sgd['final_val_loss'].idxmin()]
        ax.plot(epochs, get_loss_curve(best_sgd_row, 'train'), label=f"SGD ($\eta={lr_labels[best_sgd_row['lr']]}$)", color='k', linestyle='--', linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Train Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, 1D Random Polynomial ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.ylim([None,3])
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"train_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

In [ ]:
for LR in [0.05,0.1,0.5,1.0]:
    for RTOL in [1e-4,1e-3,1e-2]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            train_curve = get_loss_curve(row, 'val')
            epochs = np.arange(len(train_curve))
            ax.plot(epochs, train_curve, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        best_sgd_row = df_baseline.loc[df_sgd['final_val_loss'].idxmin()]
        ax.plot(epochs, get_loss_curve(best_sgd_row, 'val'), label=f"SGD ($\eta={lr_labels[best_sgd_row['lr']]}$)", color='k', linestyle='--', linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, Polynomial ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.ylim([None,3])
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"val_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

# SVD vs baseline - randomized $k$-SVD version 2 (eigenvalues of JJ^T)

In [ ]:
# Load JSONL results
from style import load_results, load_results_jsonl

PLOT_DIR = Path('plots/polynomial_SVDrandv2')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

df = load_results("polynomial_scan")

def is_bad(row):
    if np.isnan(row['losses']['val'][-1]):
        return True
    return False
df['is_bad'] = df.apply(is_bad, axis=1)
df = df[~df['is_bad']]
df_baseline = df[df['optimizer'] != 'SVD'].copy()
del df

## load new SVD results
df_svd = load_results("polynomial_scan_SVDrandv2")

In [ ]:
SEED_SEL = 5310
BASELINE_OPT = ['SGD','PolyakSGD','RMSprop','Adam','LBFGS','JD_UPGrad','HIG']
df_baseline = df_baseline[df_baseline.model_seed == SEED_SEL]
df_svd = df_svd[df_svd.model_seed == SEED_SEL]

In [ ]:
# Helper functions
def get_final_loss(row, loss_type='val'):
    losses = row['losses'][loss_type]
    # Return last non-NaN value (handles LBFGS instability)
    for val in reversed(losses):
        if val is not None and not (isinstance(val, float) and np.isnan(val)):
            return val
    return np.nan

def get_loss_curve(row, loss_type='val'):
    return np.array(row['losses'][loss_type])

def sliding_average(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

# Add derived columns
df_baseline['final_val_loss'] = df_baseline.apply(lambda r: get_final_loss(r, 'val'), axis=1)
df_baseline['final_train_loss'] = df_baseline.apply(lambda r: get_final_loss(r, 'train'), axis=1)
df_baseline['total_time'] = df_baseline['losses'].apply(lambda l: l.get('total_time', np.nan))
df_baseline['avg_epoch_time'] = df_baseline['losses'].apply(lambda l: l.get('avg_epoch_time', np.nan))
df_baseline['avg_batch_time_train'] = df_baseline['losses'].apply(lambda l: l.get('avg_batch_time_train', np.nan))

df_svd['final_val_loss'] = df_svd.apply(lambda r: get_final_loss(r, 'val'), axis=1)
df_svd['final_train_loss'] = df_svd.apply(lambda r: get_final_loss(r, 'train'), axis=1)
df_svd['total_time'] = df_svd['losses'].apply(lambda l: l.get('total_time', np.nan))
df_svd['avg_epoch_time'] = df_svd['losses'].apply(lambda l: l.get('avg_epoch_time', np.nan))
df_svd['avg_batch_time_train'] = df_svd['losses'].apply(lambda l: l.get('avg_batch_time_train', np.nan))

bs = sorted(df_svd['batch_size'].unique())[0]  # 32
baseline_optimizers = sorted(df_baseline['optimizer'].unique().tolist())
k_fractions = sorted(df_svd['k_fraction'].dropna().unique())
svd_lrs = sorted(df_svd['lr'].unique())
svd_rtols = sorted(df_svd['rtol'].dropna().unique())

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
n_epochs = len(train_curve)
epochs_train = np.arange(1,n_epochs + 1)

ax.plot(epochs_train, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]
    label = f"{opt}"
    ax.plot(epochs_train, get_loss_curve(best_row, 'train'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,5])
plt.xlim(0,n_epochs+1)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'train_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'val')
n_epochs = len(train_curve)
epochs_train = np.arange(n_epochs)

ax.plot(epochs_train, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]

    label = f"{opt}"
    ax.plot(epochs_train, get_loss_curve(best_row, 'val'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
plt.xlim(0,n_epochs)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'val_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
wall_times = get_loss_curve(best_svd_row, 'epoch_times')
t = np.cumsum(wall_times)

ax.plot(t, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]
    t = get_loss_curve(best_row, 'epoch_times')
    t = np.cumsum(t)
    label = f"{opt}"
    ax.plot(t, get_loss_curve(best_row, 'train'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Wall Time (s)')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
#plt.xlim([1,None])
plt.xscale('log')
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_train_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df_svd.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'val')
wall_times = get_loss_curve(best_svd_row, 'epoch_times')
wall_times = np.concatenate([np.array([1]), np.array(wall_times)])
t = np.cumsum(wall_times)

ax.plot(t, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(BASELINE_OPT):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df_baseline.loc[opt_df['final_val_loss'].idxmin()]
    t = get_loss_curve(best_row, 'epoch_times')
    t = np.concatenate([np.array([1]), np.array(t)])
    t = np.cumsum(t)
    #label = f"{opt} ($\eta={lr_labels[best_row['lr']]}$)" if opt != "PolyakSGD" else f"{opt}"
    label = f"{opt}"
    ax.plot(t, get_loss_curve(best_row, 'val'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Wall Time + 1 (sec)')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
plt.xlim([1,None])
plt.xscale('log')
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_val_loss_best.pdf')
plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
    for RTOL in [1e-4,1e-3,1e-2]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            train_curve = get_loss_curve(row, 'train')
            epochs = np.arange(1, len(train_curve) + 1)
            ax.plot(epochs, train_curve, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        best_sgd_row = df_baseline.loc[df_sgd['final_val_loss'].idxmin()]
        ax.plot(epochs, get_loss_curve(best_sgd_row, 'train'), label=f"SGD ($\eta={lr_labels[best_sgd_row['lr']]}$)", color='k', linestyle='--', linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Train Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, 1D Random Polynomial ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.ylim([None,3])
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"train_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
    for RTOL in [1e-4,1e-3,1e-2]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            train_curve = get_loss_curve(row, 'val')
            epochs = np.arange(len(train_curve))
            ax.plot(epochs, train_curve, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        best_sgd_row = df_baseline.loc[df_sgd['final_val_loss'].idxmin()]
        ax.plot(epochs, get_loss_curve(best_sgd_row, 'val'), label=f"SGD ($\eta={lr_labels[best_sgd_row['lr']]}$)", color='k', linestyle='--', linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, Polynomial ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.ylim([None,3])
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"val_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()